# YOLOv8 Deteksi Cacat PCB DeepPCB
By : Ashif Dyan Armawan


# FASE 1: SETUP LINGKUNGAN & IMPORT PUSTAKA


In [ ]:
!pip install ultralytics opencv-python-headless matplotlib pandas scikit-learn

In [ ]:
import os
import cv2
import yaml
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

In [ ]:
# Definisi Path Kritis
INPUT_DIR = "/kaggle/input/datasets/yidazhang07/bridge-cracks-image/DeepPCB/PCBData" 
WORKING_DIR = "/kaggle/working/"
DATASET_YOLO_DIR = os.path.join(WORKING_DIR, "datasets", "deeppcb")
IMG_SIZE = 640

# Pemetaan Kelas (Basis-0 untuk YOLO)
CLASS_NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']

print("[INFO] CELL 1 Selesai: Dependensi siap dan Variabel Global telah diamankan dalam memori kernel.")

# FASE 2: DATA MAPPING & VALIDASI

In [ ]:
def build_mixed_dataset(raw_images, raw_txts):
    print("\n[INFO] Memetakan Graf Bipartit: Integrasi Cacat (Positif) & Background (Negatif)...")
    txt_dict = {txt.stem: txt for txt in raw_txts}
    txt_dict_clean = {txt.stem.replace('_test', ''): txt for txt in raw_txts}
    
    positive_pairs = [] # Berisi cacat
    negative_pairs = [] # Berisi background murni
    
    for img in raw_images:
        # KATEGORI 1: GAMBAR BACKGROUND (Tanpa Cacat)
        if img.stem.endswith('_temp'):
            # Kita tandai dengan 'None' karena tidak ada file txt aslinya
            negative_pairs.append((img, None)) 
            continue
            
        # KATEGORI 2: GAMBAR CACAT
        if img.stem in txt_dict:
            positive_pairs.append((img, txt_dict[img.stem]))
        else:
            clean_stem = img.stem.replace('_test', '')
            if clean_stem in txt_dict_clean:
                positive_pairs.append((img, txt_dict_clean[clean_stem]))

    print(f" -> Ditemukan {len(positive_pairs)} sampel Cacat.")
    print(f" -> Ditemukan {len(negative_pairs)} sampel Background Murni.")
    
    # Rasio Emas: Gabungkan semua positif dengan background
    # (DeepPCB memiliki rasio 1:1 antara _test dan _temp, ini sangat ideal)
    return positive_pairs + negative_pairs

def convert_to_yolo_format_safe(txt_path_in, txt_path_out):
    """Fungsi konversi yang aman. Jika txt_path_in adalah None, buat file kosong."""
    # Skenario Background: Buat file .txt kosong total
    if txt_path_in is None:
        open(txt_path_out, 'w').close()
        return

    # Skenario Normal: Konversi koordinat
    with open(txt_path_in, 'r') as f_in, open(txt_path_out, 'w') as f_out:
        for line in f_in:
            parts = line.strip().split()
            if len(parts) < 5: continue
            x1, y1, x2, y2, cls_type = map(float, parts[:5])
            
            x_min, x_max = min(x1, x2), max(x1, x2)
            y_min, y_max = min(y1, y2), max(y1, y2)
            
            x_center = np.clip(((x_min + x_max) / 2.0) / IMG_SIZE, 0.0, 1.0)
            y_center = np.clip(((y_min + y_max) / 2.0) / IMG_SIZE, 0.0, 1.0)
            width = np.clip((x_max - x_min) / IMG_SIZE, 0.0, 1.0)
            height = np.clip((y_max - y_min) / IMG_SIZE, 0.0, 1.0)
            
            f_out.write(f"{int(cls_type) - 1} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

def process_and_inject_dataset(mixed_pairs):
    print("\n[INFO] Mengeksekusi Standardisasi Domain & Pembuatan File Kosong (Negative Mining)...")
    
    # Bersihkan direktori lama agar tidak tercampur
    if os.path.exists(DATASET_YOLO_DIR):
        shutil.rmtree(DATASET_YOLO_DIR)
        
    for split in ['train', 'val']:
        os.makedirs(os.path.join(DATASET_YOLO_DIR, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(DATASET_YOLO_DIR, 'labels', split), exist_ok=True)

    # Pisahkan dataset menjadi Train dan Val
    train_pairs, val_pairs = train_test_split(mixed_pairs, test_size=0.2, random_state=42)
    
    positive_count, negative_count = 0, 0
    
    for split_name, pair_list in zip(['train', 'val'], [train_pairs, val_pairs]):
        for img_path, txt_path in pair_list:
            # 1. Standardisasi Domain (Grayscale 3-Channel)
            img_array = cv2.imread(str(img_path))
            if img_array is None: continue
                
            gray_img = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
            gray_3_channel = cv2.merge([gray_img, gray_img, gray_img])
            
            # 2. Penamaan Destinasi
            dest_img = os.path.join(DATASET_YOLO_DIR, 'images', split_name, img_path.name)
            dest_txt = os.path.join(DATASET_YOLO_DIR, 'labels', split_name, f"{img_path.stem}.txt")
            
            # 3. Simpan Gambar & Proses Label (kosong atau berisi)
            cv2.imwrite(dest_img, gray_3_channel)
            convert_to_yolo_format_safe(txt_path, dest_txt)
            
            if txt_path is None: negative_count += 1
            else: positive_count += 1
            
    print(f"[INFO] Injeksi Selesai. Total Data Latih/Validasi: {positive_count} Gambar Cacat | {negative_count} Gambar Background")
    print("[INFO] Model Anda sekarang akan belajar membedakan mana noise dan mana cacat riil.")

# ==============================================================================
# EKSEKUSI PEMBANGUNAN ULANG DATASET
# ==============================================================================
raw_images = list(Path(INPUT_DIR).rglob('*.jpg')) 
raw_txts = list(Path(INPUT_DIR).rglob('*.txt'))

if raw_images and raw_txts:
    mixed_pairs = build_mixed_dataset(raw_images, raw_txts)
    process_and_inject_dataset(mixed_pairs)
    print("\n[SIAP] Dataset baru telah diformat. Anda dapat menjalankan ulang blok Training (model.train) Anda.")

process_and_inject_dataset(mixed_pairs)

# FASE 3 : EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
def run_eda(pairs, num_samples=3):
    print("\n[INFO] Menjalankan Fase 2: Rendering Inspeksi Visual Ekstraktif...")
    actual_samples = min(num_samples, len(pairs))
    if actual_samples == 0:
        print(" Reservoir data kosong. Terminasi visual.")
        return
        
    fig, axes = plt.subplots(1, actual_samples, figsize=(5 * actual_samples, 5))
    if actual_samples == 1: axes = [axes]
    samples = random.sample(pairs, actual_samples)
    
    for i, (img_path, txt_path) in enumerate(samples):
        img = cv2.imread(str(img_path))
        if img is None: continue
        
        # Koreksi Spektrum BGR ke RGB untuk fidelitas penampil visual
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = axes[i]
        ax.imshow(img)
        ax.set_title(f"Target: {img_path.name}")
        ax.axis('off')
        
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    x1, y1, x2, y2, cls_type = map(float, parts[:5])
                    cls_name = CLASS_NAMES[int(cls_type) - 1]
                    rect = patches.Rectangle(
                        (x1, y1), x2 - x1, y2 - y1, 
                        linewidth=2, edgecolor='red', facecolor='none'
                    )
                    ax.add_patch(rect)
                    ax.text(x1, y1 - 5, cls_name, color='red', fontsize=10, 
                            bbox=dict(facecolor='white', alpha=0.5, pad=0))
    plt.tight_layout()
    plt.show()

# Eksekusi Mandiri CELL 3
if 'valid_pairs' in locals() and len(valid_pairs) > 0:
    run_eda(valid_pairs, num_samples=3)
else:
    print(" Reservoir valid_pairs tidak tersedia di kernel memori. Operasikan Cell 2 terlebih dahulu.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import random

def run_class_specific_eda(pairs, num_samples_per_class=3):
    print("\n[INFO] Menjalankan Fase 2: Rendering Inspeksi Visual Ekstraktif (Berbasis Kelas)...")
    
    if not pairs:
        print("[ERROR] Reservoir data kosong. Terminasi visual.")
        return

    # 1. Membangun Inverted Index (Pemetaan Kelas -> Daftar Gambar)
    # Dictionary comprehension untuk menyiapkan wadah kosong bagi setiap kelas
    class_to_pairs = {i: [] for i in range(len(CLASS_NAMES))}
    
    for img_path, txt_path in pairs:
        classes_in_this_file = set() # Menggunakan set untuk menghindari duplikasi id kelas dalam 1 file
        try:
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # Hati-hati: Anotasi asli DeepPCB menggunakan 1-6, kita konversi ke 0-5
                        cls_type = int(float(parts[4])) - 1 
                        if 0 <= cls_type < len(CLASS_NAMES):
                            classes_in_this_file.add(cls_type)
        except Exception as e:
            continue
            
        # Masukkan pasangan path ini ke dalam dictionary kelas yang sesuai
        for cls_id in classes_in_this_file:
            class_to_pairs[cls_id].append((img_path, txt_path))

    # 2. Proses Rendering per Kelas
    for cls_id, cls_name in enumerate(CLASS_NAMES):
        available_pairs = class_to_pairs[cls_id]
        actual_samples = min(num_samples_per_class, len(available_pairs))
        
        print(f"\n--- Merender Kelas: [{cls_name.upper()}] | Populasi Tersedia: {len(available_pairs)} gambar ---")
        
        if actual_samples == 0:
            print(f" [WARNING] Tidak ada sampel gambar untuk kelas {cls_name} di reservoir data saat ini.")
            continue
            
        fig, axes = plt.subplots(1, actual_samples, figsize=(5 * actual_samples, 5))
        if actual_samples == 1: axes = [axes] # Konversi ke list jika hanya 1 sampel agar bisa diiterasi
        
        samples = random.sample(available_pairs, actual_samples)
        
        for i, (img_path, txt_path) in enumerate(samples):
            img = cv2.imread(str(img_path))
            if img is None: continue
            
            # Koreksi Spektrum BGR ke RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax = axes[i]
            ax.imshow(img)
            ax.set_title(f"Target: {img_path.name}")
            ax.axis('off')
            
            # Gambar ulang semua Bounding Box di gambar tersebut
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        x1, y1, x2, y2, cls_type_raw = map(float, parts[:5])
                        current_cls_id = int(cls_type_raw) - 1
                        current_cls_name = CLASS_NAMES[current_cls_id]
                        
                        # Jika box ini adalah kelas target utama yang sedang kita cari, beri warna MERAH TEBAL
                        # Jika box ini adalah kelas lain yang kebetulan ada di gambar yang sama, beri warna KUNING TIPIS
                        is_target_class = (current_cls_id == cls_id)
                        box_color = 'red' if is_target_class else 'yellow'
                        line_width = 3 if is_target_class else 1
                        alpha_val = 1.0 if is_target_class else 0.5
                        
                        rect = patches.Rectangle(
                            (x1, y1), x2 - x1, y2 - y1, 
                            linewidth=line_width, edgecolor=box_color, facecolor='none', alpha=alpha_val
                        )
                        ax.add_patch(rect)
                        ax.text(x1, y1 - 5, current_cls_name, color=box_color, fontsize=9, 
                                bbox=dict(facecolor='black', alpha=0.7, pad=1))
        
        plt.tight_layout()
        plt.show()

# Eksekusi Mandiri CELL 3
if 'valid_pairs' in locals() and len(valid_pairs) > 0:
    run_class_specific_eda(valid_pairs, num_samples_per_class=3)
else:
    print(" Reservoir valid_pairs tidak tersedia di kernel memori. Operasikan Cell 2 terlebih dahulu.")

# FASE 4 : YOLO PREPROCESSING & KONVERSI DATASET

In [ ]:
def setup_yolo_directories():
    for split in ['train', 'val']:
        os.makedirs(os.path.join(DATASET_YOLO_DIR, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(DATASET_YOLO_DIR, 'labels', split), exist_ok=True)

def convert_to_yolo_format_safe(txt_path_in, txt_path_out):
    if txt_path_in is None:
        open(txt_path_out, 'w').close() # Buat file kosong
        return

    with open(txt_path_in, 'r') as f_in, open(txt_path_out, 'w') as f_out:
        for line in f_in:
            parts = line.strip().split()
            if len(parts) < 5: continue
            x1, y1, x2, y2, cls_type = map(float, parts[:5])
            
            x_min, x_max = min(x1, x2), max(x1, x2)
            y_min, y_max = min(y1, y2), max(y1, y2)
            
            xc = np.clip(((x_min + x_max) / 2.0) / IMG_SIZE, 0.0, 1.0)
            yc = np.clip(((y_min + y_max) / 2.0) / IMG_SIZE, 0.0, 1.0)
            w = np.clip((x_max - x_min) / IMG_SIZE, 0.0, 1.0)
            h = np.clip((y_max - y_min) / IMG_SIZE, 0.0, 1.0)
            
            f_out.write(f"{int(cls_type) - 1} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

def process_and_inject_dataset(mixed_pairs):
    print("\n[INFO] Fase 2: Standardisasi Domain & Pembuatan File Kosong (Negative Mining)...")
    setup_yolo_directories()
    
    train_pairs, val_pairs = train_test_split(mixed_pairs, test_size=0.2, random_state=42)
    valid_count, anomaly_count = 0, 0
    
    for split_name, pair_list in zip(['train', 'val'], [train_pairs, val_pairs]):
        for img_path, txt_path in pair_list:
            img_array = cv2.imread(str(img_path))
            
            if is_image_corrupted(img_array):
                anomaly_count += 1
                continue
                
            gray_img = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
            gray_3_channel = cv2.merge([gray_img, gray_img, gray_img])
            
            dest_img = os.path.join(DATASET_YOLO_DIR, 'images', split_name, img_path.name)
            dest_txt = os.path.join(DATASET_YOLO_DIR, 'labels', split_name, f"{img_path.stem}.txt")
            
            cv2.imwrite(dest_img, gray_3_channel)
            convert_to_yolo_format_safe(txt_path, dest_txt)
            valid_count += 1
            
    print(f"[INFO] Pipeline Preprocessing Selesai. Data Masuk: {valid_count} | Dibuang (Sensor Mati): {anomaly_count}")

def create_yaml():
    yaml_path = os.path.join(WORKING_DIR, 'data.yaml')
    yaml_content = {
        'train': os.path.join(DATASET_YOLO_DIR, 'images', 'train'),
        'val': os.path.join(DATASET_YOLO_DIR, 'images', 'val'),
        'nc': len(CLASS_NAMES),
        'names': CLASS_NAMES
    }
    with open(yaml_path, 'w') as f: yaml.dump(yaml_content, f, sort_keys=False)
    return yaml_path

# FASE 5 : YOLO TRAINING

In [ ]:
def train_yolo(config_path):
    print("\n[INFO] Memicu Mesin Eksekusi Pelatihan YOLOv8 Arsitektur Nano...")
    print(" Sistem membutuhkan verifikasi status alokasi akselerator keras GPU Kaggle.")
    
    # Inisialisasi arsitektur pratelatih
    model = YOLO('yolov8n.pt') 
    
    # Pengikatan loop latih dengan delegasi ke prosesor Tensor GPU
    results = model.train(
        data=config_path, 
        epochs=50, 
        imgsz=IMG_SIZE, 
        batch=16,
        project=os.path.join(WORKING_DIR, 'runs'), 
        name='deeppcb_detect', 
        exist_ok=True
    )
    return model, results

# Eksekusi Mandiri CELL 5
if 'yaml_path' in locals() and os.path.exists(yaml_path):
    trained_model, training_results = train_yolo(yaml_path)
else:
    print(" Referensi konfigurasi manifest yaml_path rusak. Komputasi dihentikan paksa.")

# FASE 6 : EVALUASI METRIK PASKAPELATIHAN

In [ ]:

def show_evaluation_metrics():
    print("\n[INFO] Mengurai Ekstraksi Skalar Berorientasi Integritas Manufaktur...")
    cm_path = os.path.join(WORKING_DIR, 'runs', 'deeppcb_detect', 'confusion_matrix.png')
    results_csv = os.path.join(WORKING_DIR, 'runs', 'deeppcb_detect', 'results.csv')
    
    if os.path.exists(cm_path):
        cm_img = cv2.imread(cm_path)
        plt.figure(figsize=(10, 10))
        plt.imshow(cv2.cvtColor(cm_img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title("Visualisasi Matriks Konfusi (Identifikasi Bias Toleransi Mode Kegagalan)")
        plt.show()
    else:
        print("[WARNING] Log matriks grafis tidak terekam dalam disk.")
        
    if os.path.exists(results_csv):
        import pandas as pd
        df = pd.read_csv(results_csv)
        
        # Ekstraksi matriks konvergensi akhir log eksekusi
        last_epoch = df.iloc[-1]
        print("\n--- AGREGASI METRIK ITERASI TERMINAL ---")
        
        # Menghapus spasi ekstra pada nama kolom jika ada (pembersihan data standar)
        cols = [col.strip() for col in df.columns] 
        df.columns = cols
        
        try:
            # PERBAIKAN: Melengkapi list comprehension dan menambahkan [0] untuk ekstraksi skalar (string tunggal)
            recall_col = [c for c in cols if 'recall' in c.lower()][0]
            map50_col = [c for c in cols if 'mAP50(B)' in c][0]
            map95_col = [c for c in cols if 'mAP50-95(B)' in c][0]
            
            print(f"Proporsi Recall (Sensitivitas Deteksi Cacat) : {last_epoch[recall_col]:.4f}")
            print(f"Rata-rata Presisi Rerata (mAP@50)            : {last_epoch[map50_col]:.4f}")
            print(f"Rata-rata Presisi Rerata Rentang (mAP@50-95) : {last_epoch[map95_col]:.4f}")
        except IndexError:
            # Jika [0] gagal karena list kosong (nama kolom berubah drastis), block ini akan menangkapnya
            print("[ERROR] Pergeseran paradigma nama kolom internal library YOLO. Pemeriksaan manual `results.csv` diwajibkan.")
            print(f"Kolom yang tersedia saat ini: {cols}")
    else:
        print("[WARNING] Metadata metrik pelatihan skalar tidak diekspor oleh mesin log. Apakah training selesai?")

# Eksekusi Mandiri CELL 6
show_evaluation_metrics()

In [ ]:
def visualize_background_hallucinations(conf_threshold=0.25):
    print(f"\n[INFO] Mengekstraksi perbandingan Fakta vs Halusinasi Model (Conf: {conf_threshold})...")
    best_weights = os.path.join(WORKING_DIR, 'runs', 'deeppcb_detect', 'weights', 'best.pt')
    if not os.path.exists(best_weights):
        print("[ERROR] Bobot model tidak ditemukan.")
        return

    model = YOLO(best_weights)
    val_images_dir = Path(os.path.join(DATASET_YOLO_DIR, 'images', 'val'))
    val_labels_dir = Path(os.path.join(DATASET_YOLO_DIR, 'labels', 'val'))
    
    val_images = list(val_images_dir.glob('*.jpg'))
    if not val_images: return

    # Ambil 4 sampel acak untuk analisis
    samples = random.sample(val_images, min(4, len(val_images)))
    
    for img_path in samples:
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        ax.imshow(img)
        
        # 1. GAMBAR GROUND TRUTH (FAKTA) - WARNA HIJAU
        txt_path = val_labels_dir / f"{img_path.stem}.txt"
        has_gt = False
        if txt_path.exists():
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        has_gt = True
                        cls_id, xc, yc, bw, bh = map(float, parts[:5])
                        # Konversi YOLO norm ke Absolut
                        x1 = (xc - bw/2) * w
                        y1 = (yc - bh/2) * h
                        box_w, box_h = bw * w, bh * h
                        
                        rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor='lime', facecolor='none', linestyle='--')
                        ax.add_patch(rect)
                        ax.text(x1, y1 - 10, f"GT: {CLASS_NAMES[int(cls_id)]}", color='lime', fontsize=9, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, pad=2))

        # 2. GAMBAR PREDIKSI YOLO (TEBAKAN) - WARNA MERAH
        results = model.predict(source=str(img_path), imgsz=IMG_SIZE, conf=conf_threshold, verbose=False)[0]
        
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0].cpu().numpy())
            conf = float(box.conf[0].cpu().numpy())
            
            box_w, box_h = x2 - x1, y2 - y1
            rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y2 + 15, f"Pred: {CLASS_NAMES[cls_id]} {conf:.2f}", color='red', fontsize=9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, pad=2))

        ax.axis('off')
        
        # Kesimpulan Visual
        title = f"Analisis: {img_path.name}\n"
        title += "[HIJAU Putus-putus = Fakta (Ground Truth)] | [MERAH Solid = Prediksi Model]\n"
        title += "CARA BACA: Jika ada kotak MERAH berdiri sendiri tanpa kotak HIJAU di dekatnya,\nitu adalah False Positive (Noise Background)."
        
        plt.title(title, fontsize=10, pad=10)
        plt.tight_layout()
        plt.show()

# Eksekusi Mandiri
visualize_background_hallucinations(conf_threshold=0.25)

# FASE 7 : INFERENSI VISUAL PADA DATA VALIDASI

In [ ]:
def run_inference():
    print("\n[INFO] Meluncurkan Orkestrasi Inferensi Pasif pada Kumpulan Sampel Unseen...")
    best_weights = os.path.join(WORKING_DIR, 'runs', 'deeppcb_detect', 'weights', 'best.pt')
    if not os.path.exists(best_weights): 
        print(f" Fail seri bobot komputasi 'best.pt' tidak terekam pada lintasan komputasional: {best_weights}")
        return
        
    model = YOLO(best_weights)
    val_dir = os.path.join(DATASET_YOLO_DIR, 'images', 'val')
    val_images = list(Path(val_dir).glob('*.jpg'))
    
    if not val_images: 
        print(f" Array input data visual tidak eksisten dalam subdirektori lintas pengujian: {val_dir}")
        return
        
    for img_path in random.sample(val_images, min(5, len(val_images))):
        # PERBAIKAN: Tambahkan indeks [0] untuk mengekstrak objek Results dari dalam list kembalian
        res_results = model.predict(source=str(img_path), imgsz=IMG_SIZE, conf=0.25)
        res_plotted = res_results[0].plot() # Ekstraksi indeks pertama sebelum di-plot

        plt.figure(figsize=(8, 8))
        # Ganti variabel 'res' dengan 'res_plotted'
        plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)) 
        plt.title(f"Anotasi Prediktif Model Jaringan Ekstrak YOLOv8: {img_path.name}")
        plt.axis('off')
        plt.show()

# Eksekusi Mandiri CELL 7
run_inference()